# Final Class SQL Review: Database Administration Ideas With SQLite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_15/final_class_sql_review_notebook.ipynb)

This notebook reviews relational database administration concepts from the course.

It uses SQLite because SQLite runs directly inside Colab without a cloud account. The ideas transfer to PostgreSQL and Supabase:

- schema design
- primary keys and foreign keys
- constraints
- inserts and seed data
- joins
- indexes
- transactions
- rollback
- backup-style export

SQLite is not PostgreSQL. Some syntax and internals differ. Use this notebook as a concept review, not as a complete Postgres substitute.


## 1. Create A Small Database

Imagine a final project about campus equipment checkout.

The database needs to track:

- students
- equipment
- checkout records

A database administrator should care about table structure, keys, constraints, and whether the design can answer real questions.


In [ ]:
import sqlite3
from pathlib import Path

# Use a file instead of an in-memory database so we can demonstrate a backup-style export later.
db_path = Path("final_class_equipment_review.sqlite")

# Make the notebook repeatable. This only deletes the local demo file.
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# Foreign keys are off by default in SQLite, so we explicitly enable them.
cur.execute("PRAGMA foreign_keys = ON;")

print("Created database:", db_path)


## 2. Define Tables With Constraints

A constraint is a rule the database enforces.

Examples:

- primary key: each row has a stable identifier
- foreign key: a value must match a row in another table
- not null: a column must have a value
- check: a value must be from an allowed set

Constraints are part of administration because they protect data quality.


In [ ]:
cur.executescript("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE
);

CREATE TABLE equipment (
    equipment_id INTEGER PRIMARY KEY,
    equipment_name TEXT NOT NULL,
    category TEXT NOT NULL CHECK (category IN ('laptop', 'camera', 'microphone', 'adapter')),
    is_active INTEGER NOT NULL CHECK (is_active IN (0, 1))
);

CREATE TABLE checkouts (
    checkout_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    equipment_id INTEGER NOT NULL,
    checkout_date TEXT NOT NULL,
    due_date TEXT NOT NULL,
    return_date TEXT,
    status TEXT NOT NULL CHECK (status IN ('checked_out', 'returned', 'overdue')),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (equipment_id) REFERENCES equipment(equipment_id)
);
""")

conn.commit()
print("Created students, equipment, and checkouts tables")


## 3. Insert Seed Data

Seed data proves that the model can hold realistic examples.

For a final project, seed data should be large enough to test real queries, but small enough to inspect while learning.


In [ ]:
students = [
    (1, "Avery Rivera", "avery@example.edu"),
    (2, "Jordan Lee", "jordan@example.edu"),
    (3, "Morgan Patel", "morgan@example.edu"),
]

equipment = [
    (1, "Dell Latitude 5440", "laptop", 1),
    (2, "Canon M50", "camera", 1),
    (3, "Blue Yeti USB Mic", "microphone", 1),
    (4, "USB-C HDMI Adapter", "adapter", 1),
]

checkouts = [
    (1, 1, 1, "2026-05-01", "2026-05-08", None, "checked_out"),
    (2, 2, 2, "2026-05-03", "2026-05-10", "2026-05-09", "returned"),
    (3, 1, 4, "2026-05-06", "2026-05-13", None, "overdue"),
    (4, 3, 3, "2026-05-07", "2026-05-14", None, "checked_out"),
]

cur.executemany("INSERT INTO students VALUES (?, ?, ?);", students)
cur.executemany("INSERT INTO equipment VALUES (?, ?, ?, ?);", equipment)
cur.executemany("INSERT INTO checkouts VALUES (?, ?, ?, ?, ?, ?, ?);", checkouts)
conn.commit()

print("Inserted seed data")


## 4. Write A Useful Join Query

A join combines related rows.

This query answers a realistic admin question:

> Which items are currently checked out or overdue, and who has them?


In [ ]:
active_checkout_query = """
SELECT
    checkouts.checkout_id,
    students.full_name,
    equipment.equipment_name,
    equipment.category,
    checkouts.checkout_date,
    checkouts.due_date,
    checkouts.status
FROM checkouts
JOIN students ON checkouts.student_id = students.student_id
JOIN equipment ON checkouts.equipment_id = equipment.equipment_id
WHERE checkouts.status IN ('checked_out', 'overdue')
ORDER BY checkouts.due_date ASC;
"""

rows = cur.execute(active_checkout_query).fetchall()
for row in rows:
    print(dict(row))


## 5. Add An Index For A Real Access Pattern

Indexes help the database find rows faster.

A good index is tied to a real query pattern. Here, the query filters by `status` and sorts by `due_date`.

Tradeoff: indexes improve some reads, but they use storage and add maintenance work during writes.


In [ ]:
# Inspect the query plan before adding the index.
print("Before index:")
for row in cur.execute("EXPLAIN QUERY PLAN " + active_checkout_query):
    print(tuple(row))

# Create an index that matches the filter and sort pattern.
cur.execute("CREATE INDEX idx_checkouts_status_due_date ON checkouts (status, due_date);")
conn.commit()

print("\nAfter index:")
for row in cur.execute("EXPLAIN QUERY PLAN " + active_checkout_query):
    print(tuple(row))


## 6. Demonstrate A Transaction And Rollback

A transaction lets multiple actions succeed or fail together.

This matters because database administrators need to avoid partial, inconsistent changes.


In [ ]:
# Start a transaction manually.
conn.isolation_level = None
cur.execute("BEGIN;")

# Pretend we are about to mark an overdue item as returned.
cur.execute("""
UPDATE checkouts
SET status = 'returned', return_date = '2026-05-19'
WHERE checkout_id = 3;
""")

print("Inside the transaction:")
print(dict(cur.execute("SELECT checkout_id, status, return_date FROM checkouts WHERE checkout_id = 3;").fetchone()))

# We decide this was a mistake, so we roll it back.
cur.execute("ROLLBACK;")

print("\nAfter rollback:")
print(dict(cur.execute("SELECT checkout_id, status, return_date FROM checkouts WHERE checkout_id = 3;").fetchone()))

# Restore normal connection behavior for the rest of the notebook.
conn.isolation_level = ""


## 7. Create A Backup-Style Export

In production, backup and restore depend on the database platform.

This small export is not a real production backup. It is a classroom demonstration of the idea: an administrator should be able to produce evidence of database structure and data state.


In [ ]:
backup_path = Path("final_class_equipment_review_backup.sql")

with backup_path.open("w") as backup_file:
    for line in conn.iterdump():
        backup_file.write(line + "\n")

print("Wrote backup-style SQL export:", backup_path)
print("First 12 lines:")

for i, line in enumerate(backup_path.read_text().splitlines()[:12], start=1):
    print(f"{i:02d}: {line}")


## Final Reflection

Use these prompts for your GitHub concept artifact or interview preparation:

- The relational concept I can explain best is _____.
- A database administrator would verify this by _____.
- One tradeoff is _____.
- One piece of evidence from this notebook is _____.
